In [ ]:
# Step 1: Install
!pip install opencv-python scikit-learn matplotlib

# Step 2: Upload dataset OR use simulated data directly
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Simulated image data (no dataset needed) ──
np.random.seed(42)
n = 2000  # 1000 cats + 1000 dogs
X = np.random.rand(n, 4096)  # 64x64 flattened image
y = np.array([0]*1000 + [1]*1000)  # 0=cat, 1=dog

# ── Preprocessing ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=100, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"Variance retained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# ── Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

# ── Train SVM ──
print("Training SVM... (takes 1-2 mins)")
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train, y_train)

# ── Evaluate ──
y_pred = svm.predict(X_test)
print(f"\n✅ Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))

# ── Plots ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("SVM — Cats vs Dogs", fontsize=14, fontweight='bold')

# Confusion Matrix
ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Cat','Dog']); ax.set_yticklabels(['Cat','Dog'])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=18,
                fontweight='bold', color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)

# PCA Variance
ax = axes[1]
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(cumvar, color='steelblue', lw=2)
ax.axhline(95, color='red', linestyle='--', label='95% variance')
ax.set_xlabel("Components"); ax.set_ylabel("Cumulative Variance (%)")
ax.set_title("PCA Variance Explained")
ax.legend(); ax.grid(True, alpha=0.3)

# Class Distribution
ax = axes[2]
ax.bar(['Cat', 'Dog'], [1000, 1000], color=['#3498db','#e74c3c'], edgecolor='black')
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution")

plt.tight_layout()
plt.savefig('svm_cats_dogs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Done!")